In [1]:
!pip install -q transformers torch

In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

print("Libraries imported successfully")
print("PyTorch version:", torch.__version__)

Libraries imported successfully
PyTorch version: 2.11.0+cpu


In [3]:
#Create the Text Encoder (code for Task 3)
class TextEncoder:
    def __init__(self, model_name="bert-base-uncased", max_length=32, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length

        print(f"Loading model: {model_name} on {self.device} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # we only need inference
        self.hidden_size = self.model.config.hidden_size
        print(f"Model loaded. Hidden size = {self.hidden_size}")

    @torch.no_grad()
    def encode(self, texts):
        """
        Convert text (or list of texts) into embeddings.
        Returns: tensor of shape (batch_size, hidden_size)
        """
        if isinstance(texts, str):
            texts = [texts]

        inputs = self.tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        outputs = self.model(**inputs)
        # Use [CLS] token embedding
        embeddings = outputs.last_hidden_state[:, 0, :]
        return embeddings

# Creates the encoder
encoder = TextEncoder(model_name="bert-base-uncased", max_length=32)

Loading model: bert-base-uncased on cpu ...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Hidden size = 768


In [4]:
#Testing with our own text descriptions
# Test texts (you can change these)
test_texts = [
    "a bright red rose",
    "yellow sunflower in a field",
    "purple orchid with green leaves",
    "a white square shape",
    "simple geometric circle"
]

print("Input texts:")
for t in test_texts:
    print(" -", t)

# Get embeddings
embeddings = encoder.encode(test_texts)

print("\nEmbedding shape:", embeddings.shape)
print("(This means: 5 texts × 768 dimensions)")
print("\nFirst 8 values of the first embedding:")
print(embeddings[0][:8].cpu().numpy())

Input texts:
 - a bright red rose
 - yellow sunflower in a field
 - purple orchid with green leaves
 - a white square shape
 - simple geometric circle

Embedding shape: torch.Size([5, 768])
(This means: 5 texts × 768 dimensions)

First 8 values of the first embedding:
[-0.18824211  0.04167995 -0.19879471  0.04402962 -0.25844458 -0.05060856
  0.02783651  0.281115  ]


In [5]:
#To Show tokenisation details
sample_text = "a bright red rose"
tokens = encoder.tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Original text :", sample_text)
print("Input IDs     :", tokens["input_ids"].tolist())
print("Attention mask:", tokens["attention_mask"].tolist())
print("Tokens        :", encoder.tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

Original text : a bright red rose
Input IDs     : [[101, 1037, 4408, 2417, 3123, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Attention mask: [[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Tokens        : ['[CLS]', 'a', 'bright', 'red', 'rose', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [6]:
#To Show tokenisation details
sample_text = "yellow sunflower in a field"
tokens = encoder.tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Original text :", sample_text)
print("Input IDs     :", tokens["input_ids"].tolist())
print("Attention mask:", tokens["attention_mask"].tolist())
print("Tokens        :", encoder.tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

Original text : yellow sunflower in a field
Input IDs     : [[101, 3756, 3103, 14156, 1999, 1037, 2492, 102, 0, 0, 0, 0, 0, 0, 0, 0]]
Attention mask: [[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]
Tokens        : ['[CLS]', 'yellow', 'sun', '##flower', 'in', 'a', 'field', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [7]:
#To Show tokenisation details
sample_text = "purple orchid with green leaves"
tokens = encoder.tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Original text :", sample_text)
print("Input IDs     :", tokens["input_ids"].tolist())
print("Attention mask:", tokens["attention_mask"].tolist())
print("Tokens        :", encoder.tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

Original text : purple orchid with green leaves
Input IDs     : [[101, 6379, 15573, 2007, 2665, 3727, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Attention mask: [[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Tokens        : ['[CLS]', 'purple', 'orchid', 'with', 'green', 'leaves', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [8]:
#To Show tokenisation details
sample_text = "a white square shape"
tokens = encoder.tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Original text :", sample_text)
print("Input IDs     :", tokens["input_ids"].tolist())
print("Attention mask:", tokens["attention_mask"].tolist())
print("Tokens        :", encoder.tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

Original text : a white square shape
Input IDs     : [[101, 1037, 2317, 2675, 4338, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Attention mask: [[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Tokens        : ['[CLS]', 'a', 'white', 'square', 'shape', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [9]:
#To Show tokenisation details
sample_text = "simple geometric circle"
tokens = encoder.tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Original text :", sample_text)
print("Input IDs     :", tokens["input_ids"].tolist())
print("Attention mask:", tokens["attention_mask"].tolist())
print("Tokens        :", encoder.tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

Original text : simple geometric circle
Input IDs     : [[101, 3722, 14965, 4418, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Attention mask: [[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Tokens        : ['[CLS]', 'simple', 'geometric', 'circle', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


MY OBSERVATIONS (Day 2):

1. I learned how to use Hugging Face Transformers to convert text into numerical embeddings that a machine learning model can understand.

2. I observed that the BERT model used in the notebook produces a 768-dimensional embedding for each input text.

3. I learned that BERT uses the [CLS] token, and in this notebook its embedding is taken to represent the input text.

4. I understood that padding and truncation are used to keep the input texts at a fixed length, which makes it easier for the model to process them together.

5. I also learned how text is broken down into tokens and converted into input IDs and attention masks before being passed to BERT.

6. The example texts showed me that different descriptions, such as flowers and simple shapes, can be converted into their own numerical embeddings.

7. These text embeddings will be useful later as conditioning information for the Generator, so that the generated image can follow the given text description.
